# Drone detection: GPU runtime and recovery setup

This prepares the free Colab/Google Drive training host. **It does not train a
new drone detector.** The small numerical fixture tests checkpoint recovery;
it supplies no evidence about drone recognition.

1. Select **Runtime → Change runtime type → GPU** (a free GPU is not guaranteed).
2. Edit the project folder below if needed, then **Run all** and authorize Drive.
3. Check that the final report says `passed: true`.
4. Run the final flush cell. Select **Runtime → Disconnect and delete runtime**,
   reconnect with a GPU, and run all again. The second report should also say
   `replayed_checkpoint_from_earlier_invocation: true`.

Use only **one active runtime per project folder**. Colab may disconnect without
warning. Code and dependencies are recreated locally; checkpoints and the pinned
code revision live in Drive. Drive quota and source-data permissions must be
checked before any bulk data transfer. This notebook downloads no audio.


In [ ]:
# Change this before the first run if you want a different Drive folder.
# A shared drive can use /content/drive/Shareddrives/<drive>/<folder>.
PROJECT_DRIVE_PATH = '/content/drive/MyDrive/drone-detection-v2'
REPO_URL = 'https://github.com/StoneDerek/drone-detection.git'
FIRST_REF = 'main'  # Used only when bootstrap.json does not yet exist.


In [ ]:
import hashlib
import os
import shutil
import subprocess
import uuid
from pathlib import Path

from google.colab import drive

# Stop before installing the CUDA environment if a GPU was not allocated.
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                '--format=csv'], check=True)
drive.mount('/content/drive')
MOUNT = Path('/content/drive')
if not os.path.ismount(MOUNT):
    raise RuntimeError('Google Drive is not mounted; refusing an ephemeral checkpoint folder')
PROJECT = Path(PROJECT_DRIVE_PATH).resolve()
if not PROJECT.is_relative_to(MOUNT.resolve()) or PROJECT == MOUNT.resolve():
    raise ValueError('Choose a project folder inside the mounted Google Drive')
PROJECT.mkdir(parents=True, exist_ok=True)
payload = os.urandom(1024 * 1024)
probe = PROJECT / ('write-check-' + uuid.uuid4().hex + '.bin')
try:
    probe.write_bytes(payload)
    assert hashlib.sha256(probe.read_bytes()).digest() == hashlib.sha256(payload).digest()
finally:
    probe.unlink(missing_ok=True)
print('Drive mount write/read check passed. This does not measure quota or prove cloud sync.')
print('Local disk:', shutil.disk_usage('/content'))


In [ ]:
"""Standard-library-only bootstrap, embedded verbatim in the Colab notebook."""

import hashlib
import json
import os
import re
import subprocess
import uuid
from pathlib import Path


def git_output(checkout, *arguments):
    return subprocess.check_output(['git', '-C', str(checkout), *arguments], text=True).strip()


def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def pin_checkout(project, checkout, repo_url, first_ref='main'):
    """Resolve once, then reuse that exact revision after runtime replacement.

    One active runtime per project. Refuse dirty or unrelated local checkouts.
    The saved lock hash is also checked; a pin is never silently refreshed.
    """
    project, checkout = Path(project), Path(checkout)
    project.mkdir(parents=True, exist_ok=True)
    pin_path = project / 'bootstrap.json'
    pin = json.loads(pin_path.read_text()) if pin_path.exists() else None
    if pin is not None and (pin.get('schema_version') != 1
                            or pin.get('repo_url') != repo_url
                            or not re.fullmatch(r'[0-9a-f]{40}', pin.get('revision', ''))
                            or not re.fullmatch(r'[0-9a-f]{64}', pin.get('runtime_lock_sha256', ''))):
        raise ValueError('Invalid/different project pin; choose a new project folder deliberately')
    if checkout.exists() and any(checkout.iterdir()):
        if not (checkout / '.git').exists():
            raise ValueError('Checkout folder contains unrelated files')
        if git_output(checkout, 'remote', 'get-url', 'origin') != repo_url:
            raise ValueError('Checkout has a different repository origin')
        if git_output(checkout, 'status', '--porcelain'):
            raise ValueError('Local checkout changed; preserve those changes before continuing')
    else:
        checkout.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'init', '--quiet', str(checkout)], check=True)
        subprocess.run(['git', '-C', str(checkout), 'remote', 'add', 'origin', repo_url], check=True)
    target = pin['revision'] if pin else first_ref
    subprocess.run(['git', '-C', str(checkout), 'fetch', '--quiet', '--depth', '1', 'origin', target], check=True)
    revision = git_output(checkout, 'rev-parse', 'FETCH_HEAD')
    if pin and revision != pin['revision']:
        raise ValueError('Fetched revision differs from saved project pin')
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--quiet', '--detach', revision], check=True)
    candidate = {'schema_version': 1, 'repo_url': repo_url, 'revision': revision,
                 'runtime_lock_sha256': file_hash(checkout / 'notebooks/runtime/uv.lock')}
    if pin is not None and pin != candidate:
        raise ValueError('Pinned environment lock changed')
    if pin is None:
        payload = json.dumps(candidate, indent=2, sort_keys=True).encode()
        temporary = pin_path.with_name('bootstrap.partial-' + uuid.uuid4().hex)
        temporary.write_bytes(payload)
        if pin_path.exists() and json.loads(pin_path.read_text()) != candidate:
            raise ValueError('Another bootstrap wrote a different pin; use one active runtime')
        os.replace(temporary, pin_path)
        if pin_path.read_bytes() != payload:
            raise ValueError('Project pin read-back failed')
    return candidate


REPO = Path('/content/drone-detection')
PIN = pin_checkout(PROJECT, REPO, REPO_URL, FIRST_REF)
print(json.dumps(PIN, indent=2))


The first run resolves `main` once and saves `bootstrap.json` in Drive.
Later runs fetch that exact commit and verify its environment lock. Updates to
`main` will not silently change an interrupted job. To start a newer study, use
a new project folder and retain the old checkpoints.

The CUDA environment below is separate from Colab's notebook kernel. It pins
PyTorch 2.7.1 with CUDA 12.6 and NumPy; setup fails if the allocated GPU/driver
cannot run it. All training commands run in child processes using this environment.


In [ ]:
import sys

UV_DIR = Path('/content/drone-bootstrap')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
                '--target', str(UV_DIR), 'uv==0.13.5'], check=True)
UV_ENV = dict(os.environ, PYTHONPATH=str(UV_DIR),
              UV_CACHE_DIR='/content/drone-uv-cache',
              UV_PYTHON_INSTALL_DIR='/content/drone-python')
subprocess.run([sys.executable, '-m', 'uv', 'sync', '--project',
                str(REPO / 'notebooks/runtime'), '--frozen', '--python', '3.12',
                '--extra', 'gpu'], env=UV_ENV, check=True)
PYTHON = REPO / 'notebooks/runtime/.venv/bin/python'
JOB_ENV = dict(os.environ, PYTHONPATH=str(REPO / 'src'),
               CUBLAS_WORKSPACE_CONFIG=':4096:8')
PROBE_SCRIPT = REPO / 'scripts/notebook_recovery_probe.py'
subprocess.run([str(PYTHON), str(PROBE_SCRIPT), 'preflight', '--device', 'cuda'],
               cwd=REPO, env=JOB_ENV, check=True)


The recovery check runs an uninterrupted control, then kills a separate
process after update 5. Only update 4 has a complete checkpoint. A fresh process
must restore it and reach update 12 with matching model/buffers, optimizer,
scheduler, AMP scaler, random state, data order and selection history.

On later notebook invocations it replays the original persisted update-4
checkpoint in a new directory. It does not merely read a finished result.
A different GPU can change numerical results; a mismatch stops the check rather
than passing silently. This fixture uses one data worker (`num_workers=0`);
a future loader needs its own recovery check.


In [ ]:
DURABLE = PROJECT / 'runtime-checks/cuda-v1'
LOCAL = Path('/content/drone-recovery-scratch')
subprocess.run([str(PYTHON), str(PROBE_SCRIPT), 'verify', '--device', 'cuda',
                '--durable', str(DURABLE), '--local', str(LOCAL)],
               cwd=REPO, env=JOB_ENV, check=True)
report = json.loads((DURABLE / 'probe-report.json').read_text())
assert report['passed']
print('Saved report:', DURABLE / 'probe-report.json')
print('Replayed an earlier invocation:', report['replayed_checkpoint_from_earlier_invocation'])


**Before disconnecting:** run the next cell to ask Colab to flush and
unmount Drive. Then replace the runtime and repeat the notebook. A mounted
read-back alone does not independently prove that bytes reached cloud storage.
The repeat after runtime deletion is the persistence check.

This is runtime preparation. Next come the source/label/duplicate inventory and
coverage gate in [the research plan](https://github.com/StoneDerek/drone-detection/blob/main/docs/research-plan-v2.md),
then an actual audio throughput pilot and integration of the selected CNN with
this recovery mechanism. The old development recordings are not a fresh test set.


In [ ]:
drive.flush_and_unmount()
print('Drive flush/unmount completed. You can now replace the runtime and rerun.')
